# API Fetch of Green Spaces (Espaces verts et assimilés) Data
---

**Source:** https://opendata.paris.fr/explore/?disjunctive.theme&disjunctive.publisher&disjunctive.keyword&disjunctive.modified&disjunctive.features&sort=modified

**Endpoint used:** `espaces_verts`  

**Update frequency:** Daily

## 1. API Response Check

In [ ]:
import requests
import pandas as pd
import json
from tqdm.notebook import tqdm


url  = "https://opendata.paris.fr/api/explore/v2.1/catalog/datasets"
dataset = "espaces_verts"

In [ ]:
all_results = []
limit = 100
offset = 0

# Loop to fetch all records using pagination
while True:
    endpoint = "{}/{}/records?rows={}&start={}".format(url, dataset, limit, offset)
    response = requests.get(endpoint)

    if response.status_code != 200:
        print(f"Error fetching data: {response.status_code} - {response.json().get('message', 'Unknown error')}")
        break

    data = response.json()
    current_results = data.get('results', [])
    all_results.extend(current_results)

    if len(current_results) < limit:
        # No more data to fetch, break the loop
        break

    offset += limit

# Update the 'data' variable with the combined results for consistency with subsequent cells
data = {'results': all_results, 'total_count': len(all_results)}
print(f"Successfully fetched {len(all_results)} records.")

Successfully fetched 2528 records.


## 2. Fetch data and create dataframe

In [ ]:
df_green_spaces = pd.DataFrame(all_results)


# Check length of results
if len(data["results"]) == len(all_results):
    print("Length of results is equal to records available.")
else: print("Records fetched do not match records available.")

original_length = len(df_green_spaces)

Length of results is equal to records available.


## 3. Data Preprocessing & Exploration

In [ ]:
df_green_spaces.head()

,nsq_espace_vert,nom_ev,type_ev,categorie,adresse_numero,adresse_complement,adresse_typevoie,adresse_libellevoie,adresse_codepostal,poly_area,...,id_atelier_horticole,ida3d_enb,site_villes,id_eqpt,competence,geom,url_plan,geom_x_y,last_edited_user,last_edited_date
0,13161.0,JARDINIERES EVQ DE LA RUE CRESPIN DU GAST,Décorations sur la voie publique,Jardiniere,2.0,None,RUE,CRESPIN DU GAST,75011,NaN,...,12.0,JDE12802,SV,12802,CA,"{'type': 'Feature', 'geometry': {'coordinates'...",None,"{'lon': 2.3818255187845714, 'lat': 48.86591139...",None,None
1,13077.0,JARDINIERES DU SQUARE DE LA SALAMANDRE,Décorations sur la voie publique,Jardiniere,12.0,None,SQUARE DE LA,SALAMANDRE,75020,NaN,...,46.0,70710,SV,3525,CA,"{'type': 'Feature', 'geometry': {'coordinates'...",https://b22-pr-v1-iis01.ressources.paris.mdp/M...,"{'lon': 2.4058613598651686, 'lat': 48.85790156...",None,None
2,11769.0,PROMENADE PC 12 - DAUMESNIL - SAHEL,Promenades ouvertes,Promenade,22.0,None,RUE DE,MONTEMPOIVRE,75012,11854.0,...,13.0,339447,5881,10531,CP,"{'type': 'Feature', 'geometry': {'coordinates'...",https://b22-pr-v1-iis01.ressources.paris.mdp/M...,"{'lon': 2.406751045033287, 'lat': 48.838468901...",None,None
3,237.0,SQUARE GUSTAVE MESUREUR,Promenades ouvertes,Square,105.0,None,RUE,JEANNE D ARC,75013,3195.0,...,18.0,52462,3064,3603,CA,"{'type': 'Feature', 'geometry': {'coordinates'...",https://b22-pr-v1-iis01.ressources.paris.mdp/M...,"{'lon': 2.362371217923706, 'lat': 48.833740704...",None,None
4,13147.0,JARDINS DES CHAMPS ELYSEES- SQUARE DE BERLIN- ...,Promenades ouvertes,Jardin,5.0,X,AVENUE DES,CHAMPS ELYSEES,75008,NaN,...,9.0,46225,3752,4979,CP,"{'type': 'Feature', 'geometry': {'coordinates'...",https://b22-pr-v1-iis01.ressources.paris.mdp/M...,"{'lon': 2.31067213356462, 'lat': 48.8671491238...",None,None


In [ ]:
df_green_spaces.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2528 entries, 0 to 2527
Data columns (total 31 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   nsq_espace_vert        2519 non-null   float64
 1   nom_ev                 2527 non-null   object 
 2   type_ev                2527 non-null   object 
 3   categorie              2520 non-null   object 
 4   adresse_numero         2522 non-null   float64
 5   adresse_complement     371 non-null    object 
 6   adresse_typevoie       2278 non-null   object 
 7   adresse_libellevoie    2271 non-null   object 
 8   adresse_codepostal     2525 non-null   object 
 9   poly_area              1945 non-null   float64
 10  surface_totale_reelle  1932 non-null   float64
 11  surface_horticole      1859 non-null   float64
 12  presence_cloture       2246 non-null   object 
 13  perimeter              1696 non-null   float64
 14  annee_ouverture        1794 non-null   object 
 15  anne

First drop any columns we know aren't relevant for the analysis to minimize data loss from missing values.

In [ ]:
irrelevant_cols = [
    "adresse_numero",
    "adresse_typevoie",
    "adresse_complement",
    "ancien_nom_ev",
    "annee_changement_nom",
    "adresse_libellevoie",
    "presence_cloture",
    "surface_horticole",
    "surface_totale_reelle",
    "perimeter",
    "nb_entites",
    "id_division",
    "id_atelier_horticole",
    "ida3d_enb",
    "site_villes",
    "id_eqpt",
    "competence",
    "ouvert_ferme",
    "url_plan"]

df_green_spaces = df_green_spaces.drop(columns=irrelevant_cols)

print(f"Number of columns dropped: {len(irrelevant_cols)}")
print(f"Number of columns kept: {len(df_green_spaces.columns)}")
print("Remaining columns:")
for col in df_green_spaces:
  print(f"  -  {col}")


Number of columns dropped: 19
Number of columns kept: 12
Remaining columns:
  -  nsq_espace_vert
  -  nom_ev
  -  type_ev
  -  categorie
  -  adresse_codepostal
  -  poly_area
  -  annee_ouverture
  -  annee_renovation
  -  geom
  -  geom_x_y
  -  last_edited_user
  -  last_edited_date


In [ ]:
# Variable type classification
# Classifies each column by dtype and number of unique values.
# Thresholds defined based on the actual distribution of unique values in this dataset.

categories = []

for col in df_green_spaces.columns:
    series = df_green_spaces[col].dropna()
    var_type = ""
    n_unique_val = None # Initialize n_unique_val

    if series.empty:
        var_type = "Empty"
        n_unique_val = 0
    elif series.dtype in ["int64", "float64"]:
        var_type = "Quantitative"
        n_unique_val = series.nunique()
    # Check if dtype is object and if at least one element is a dictionary
    elif series.dtype == 'object' and not series.empty and series.apply(lambda x: isinstance(x, dict)).any():
        var_type = "Dictionary"
        n_unique_val = 'N/A' # Cannot calculate meaningful unique count for unhashable types directly
    else: # Handle other object types (strings, lists, etc. that are hashable)
        n_unique_val = series.nunique()
        if n_unique_val == 1:
            var_type = "Constant (single value)"
        elif n_unique_val == 2:
            var_type = "Binary"
        elif n_unique_val <= 10:
            var_type = "Categorical — Low cardinality (3–10)"
        elif n_unique_val <= 100:
            var_type = "Categorical — Medium cardinality (11–100)"
        elif n_unique_val <= 1000:
            var_type = "Categorical — High cardinality (101–1000)"
        else:
            var_type = "Categorical — Very high cardinality (>1000)"

    categories.append({
        "Column":    col,
        "Dtype":     str(df_green_spaces[col].dtype),
        "Unique":    n_unique_val,
        "Type":      var_type,
        "Missing %": round(df_green_spaces[col].isna().mean() * 100, 1)
    })

var_type_df = pd.DataFrame(categories).sort_values(by="Missing %", ascending=False)
display(var_type_df.style.background_gradient(subset=['Missing %'], cmap='Reds'))

,Column,Dtype,Unique,Type,Missing %
11,last_edited_date,object,0,Empty,100.000000
10,last_edited_user,object,0,Empty,100.000000
7,annee_renovation,object,41,Categorical — Medium cardinality (11–100),95.800000
6,annee_ouverture,object,160,Categorical — High cardinality (101–1000),29.000000
5,poly_area,float64,1247,Quantitative,23.100000
0,nsq_espace_vert,float64,2513,Quantitative,0.400000
3,categorie,object,24,Categorical — Medium cardinality (11–100),0.300000
4,adresse_codepostal,object,27,Categorical — Medium cardinality (11–100),0.100000
9,geom_x_y,object,N/A,Dictionary,0.100000
8,geom,object,N/A,Dictionary,0.100000


### Dropping Missing values & Duplicates

Check & drop duplicates.

In [ ]:
# Identify dictionary columns from the var_type_df (created previously)
dict_cols = var_type_df[var_type_df['Type'] == 'Dictionary']['Column'].tolist()

# Get all column names except the dictionary ones
columns_to_check = [col for col in df_green_spaces.columns if col not in dict_cols]

# Show duplicate values by checking a subset of columns
num_duplicates = df_green_spaces.duplicated(subset=columns_to_check).sum()

print(f"There are {num_duplicates} duplicates in the dataset (excluding the dictionary columns).")

df_green_spaces.drop_duplicates(subset=columns_to_check, inplace=True)
print("\nIf any duplicates detected: duplicates dropped.")

There are 6 duplicates in the dataset (excluding the dictionary columns).

If any duplicates detected: duplicates dropped.


Check and manage missing values.

In [ ]:
# Analyze different missing threshholds by changing the threshhold value
# Drop all columns with set threshhold, then dropna
MISSING_THRESHOLD = 0.30

# Categorize columns
missing_ratio = df_green_spaces.isna().mean()     # fraction missing per column
cols_to_drop  = missing_ratio[missing_ratio > MISSING_THRESHOLD].index.tolist()
cols_to_keep  = missing_ratio[missing_ratio <= MISSING_THRESHOLD].index.tolist()

# Show impacts
print(f"Columns before : {df_green_spaces.shape[1]}")
print(f"Columns to drop: {len(cols_to_drop)}")
print(f"Columns to keep: {len(cols_to_keep)}")
print(f"\nDropped columns:")
for col in cols_to_drop:
    print(f"  • {col:<45} ({missing_ratio[col]*100:.1f}% missing)")

print('\n')
print("Columns to keep:")
for col in cols_to_keep:
  print(f"  • {col:<45} ({missing_ratio[col]*100:.1f}% missing)")

# Drop columns at 1.2% threshhold
columns_dropped = df_green_spaces.drop(columns=cols_to_drop)

# Then drop remaining rows with missing values
dropna_after_columns_dropped = columns_dropped.dropna(axis=0)

Columns before : 12
Columns to drop: 3
Columns to keep: 9

Dropped columns:
  • annee_renovation                              (95.9% missing)
  • last_edited_user                              (100.0% missing)
  • last_edited_date                              (100.0% missing)


Columns to keep:
  • nsq_espace_vert                               (0.4% missing)
  • nom_ev                                        (0.0% missing)
  • type_ev                                       (0.0% missing)
  • categorie                                     (0.3% missing)
  • adresse_codepostal                            (0.1% missing)
  • poly_area                                     (23.1% missing)
  • annee_ouverture                               (29.0% missing)
  • geom                                          (0.1% missing)
  • geom_x_y                                      (0.1% missing)


Now we execute the drop based on the threshhold chosen above. Here we chose 30% after seeing that the most important columns we need have less than 30% missing values.

In [ ]:
# After checking impact, drop columns
df_green_spaces = df_green_spaces.drop(columns=cols_to_drop, errors='ignore')

Since `poly_area` or `anne_ouverture` are **relevant** for us but have a relatively **high share of missing values**, we will keep these columns and do a dropna for all other columns which have a relatively **low share of missing values** (<1%).


Now with a defined subset, we will only remove rows that have `NaN` values in the columns other than `poly_area` and `annee_ouverture`, leaving rows with `NaN` in these columns untouched.

In [ ]:
# Define subset for targeted dropna
columns_for_specific_dropna = [
    "nsq_espace_vert",
    "nom_ev",
    "type_ev",
    "categorie",
    "adresse_codepostal",
    "geom",
    "geom_x_y"]

df_specific_dropna = df_green_spaces.dropna(subset=columns_for_specific_dropna)

print(f"Number of rows before dropna: {len(df_green_spaces)}")
print(f"Number of rows after subset dropna: {len(df_specific_dropna)}")

# Execute dropna
df_green_spaces = df_green_spaces.dropna(subset=columns_for_specific_dropna)


Number of rows before dropna: 2522
Number of rows after subset dropna: 2503


In [ ]:
# Check results

final_missing = df_green_spaces.isna().mean().reset_index()

print("Check that only poly_area and annee_ouverture contain missing values:\n")
# Removed the standalone 'print' statement as it was not serving a functional purpose
for col in final_missing['index']:
  print(f"  • {col:<45} ({final_missing[final_missing['index'] == col][0].item()*100:.1f}% missing)")

Check that only poly_area and annee_ouverture contain missing values:

  • nsq_espace_vert                               (0.0% missing)
  • nom_ev                                        (0.0% missing)
  • type_ev                                       (0.0% missing)
  • categorie                                     (0.0% missing)
  • adresse_codepostal                            (0.0% missing)
  • poly_area                                     (22.5% missing)
  • annee_ouverture                               (29.2% missing)
  • geom                                          (0.0% missing)
  • geom_x_y                                      (0.0% missing)


In [ ]:
# Final results of missing value management
print(f"Original length: {original_length}")
print(f"Final length: {len(df_green_spaces)}")
print(f"Percentage of records retained: {len(df_green_spaces) / original_length*100:.2f}%")

Original length: 2528
Final length: 2503
Percentage of records retained: 99.01%


Now that we have the columns we need, we change the variables to the appropriate types.

In [ ]:
df_green_spaces['nsq_espace_vert'] = df_green_spaces['nsq_espace_vert'].astype(int)
df_green_spaces['adresse_codepostal'] = df_green_spaces['adresse_codepostal'].astype(int)

### Distribution of Green Spaces by Postal Code

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Count green spaces per postal code
postal_code_counts = df_green_spaces['adresse_codepostal'].value_counts().reset_index()
postal_code_counts.columns = ['Postal_Code', 'Count']

# Sort the postal codes by count in descending order
postal_code_counts_sorted = postal_code_counts.sort_values(by='Count', ascending=False)

# Create the bar plot
plt.figure(figsize=(12, 6))
sns.barplot(x='Postal_Code', y='Count', hue='Postal_Code', data=postal_code_counts_sorted, palette='viridis', legend=False, order=postal_code_counts_sorted['Postal_Code'])
plt.title('Distribution of Green Spaces by Postal Code')
plt.xlabel('Postal Code')
plt.ylabel('Number of Green Spaces')
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
print(df_green_spaces['adresse_codepostal'].unique())

[75011 75020 75012 75013 75008 75018 75014 75019 75017 75003 75004 75010
 75005 75016 75015 75007 75009 93500 75006 93210 75001 94300 75002 92220
 94200 94320 93400]


In [ ]:
df_green_spaces.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2503 entries, 0 to 2527
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nsq_espace_vert     2503 non-null   int64  
 1   nom_ev              2503 non-null   object 
 2   type_ev             2503 non-null   object 
 3   categorie           2503 non-null   object 
 4   adresse_codepostal  2503 non-null   int64  
 5   poly_area           1939 non-null   float64
 6   annee_ouverture     1772 non-null   object 
 7   geom                2503 non-null   object 
 8   geom_x_y            2503 non-null   object 
dtypes: float64(1), int64(2), object(6)
memory usage: 195.5+ KB


## 4. Translate columns & categories

In [ ]:
# Write French-English dictionary for columns
column_translations = {
    "nsq_espace_vert" : "green_space_id",
    "nom_ev" : "green_space_name",
    "type_ev" : "green_space_type",
    "categorie" : "category",
    "adresse_codepostal" : "postal_code",
    "poly_area" : "polygon_area",
    "annee_ouverture" : "opening_year",
    "geom" : "geo_shape",
    "geom_x_y" : "geo_point"}

# Translate columns
df_green_spaces = df_green_spaces.rename(columns=column_translations)
print("Columns translated.")

Columns translated.


In [ ]:
# Write French-English dictionary for modalities in `green_space_type`

type_translations = {
    "Décorations sur la voie publique": "Street decorations",
    "Promenades ouvertes": "Open promenades",
    "Murs végétalisés": "Green walls",
    "Périphérique": "Ring road",
    "Ephémères, partagés, pédagogiques": "Temporary, shared, or educational spaces",
    "Jardinets décoratifs": "Decorative small gardens",
    "Cimetières": "Cemeteries",
    "Etablissements sportifs": "Sports facilities",
    "Jardins privatifs": "Private gardens",
    "Bois": "Woodlands"
}

# Replace with translations
df_green_spaces['green_space_type'] = df_green_spaces['green_space_type'].replace(type_translations)

# Check value counts
print("Types translated. Preview of new value counts:")
df_green_spaces['green_space_type'].value_counts()

Types translated. Preview of new value counts:


,count
green_space_type,
Street decorations,989
Open promenades,596
Green walls,344
Ring road,263
"Temporary, shared, or educational spaces",142
Decorative small gardens,117
Cemeteries,20
Sports facilities,19
Private gardens,11


In [ ]:
# Translate modalities for `category`
category_translations = {
    "Jardiniere": "Planter box",
    "Murs vegetalises": "Green walls",
    "Square": "Public square",
    "Talus": "Embankment",
    "Jardin": "Garden",
    "Jardin partage": "Community garden",
    "Jardinet": "Small garden",
    "Decoration": "Decorative space",
    "Promenade": "Promenade",
    "Plate-bande": "Flower bed",
    "Cimetière": "Cemetery",
    "Espace Vert": "Green space",
    "Parc": "Park",
    "Pelouse": "Lawn",
    "Mail": "Tree-lined walkway",
    "Jardin d'immeubles": "Residential garden",
    "Esplanade": "Esplanade",
    "Foret urbaine": "Urban forest",
    "Ile": "Island",
    "Terrain de boules": "Boules court",
    "Bois": "Woodland",
    "Arboretum": "Arboretum",
    "Archipel": "Archipelago",
    "Terre-plein": "Median strip"
}

# Translate values
df_green_spaces['category'] = df_green_spaces['category'].replace(category_translations)

# Check value counts
print("Categories translated. Preview of new value counts:")
df_green_spaces['category'].value_counts()

Categories translated. Preview of new value counts:


,count
category,
Planter box,857
Green walls,340
Public square,293
Embankment,275
Garden,214
Community garden,142
Small garden,104
Decorative space,79
Promenade,66


## 5. Preparing geo-spatial data from `geo_point` and `geo_shape`.

We want to be able to plot the green spaces on some map visualisations and are not yet sure which format we need, so we
* Extract the coordinates
* Create a geometry polygon format from the GeoJSON dictionary.

In [ ]:
# Extract values from dictionary keys 'lon' and 'lat' in df_green_spaces['geo_point']
df_green_spaces['lon'] = df_green_spaces['geo_point'].apply(lambda x: x['lon'])
df_green_spaces['lat'] = df_green_spaces['geo_point'].apply(lambda x: x['lat'])

# Display new columns
df_green_spaces[['lon', 'lat', 'geo_point']].head()

,lon,lat,geo_point
0,2.381826,48.865911,"{'lon': 2.3818255187845714, 'lat': 48.86591139..."
1,2.405861,48.857902,"{'lon': 2.4058613598651686, 'lat': 48.85790156..."
2,2.406751,48.838469,"{'lon': 2.406751045033287, 'lat': 48.838468901..."
3,2.362371,48.833741,"{'lon': 2.362371217923706, 'lat': 48.833740704..."
4,2.310672,48.867149,"{'lon': 2.31067213356462, 'lat': 48.8671491238..."


In [ ]:
# import relevant libraries for creating polygons
import json
import geopandas as gpd
from shapely.geometry import Point, shape

In [ ]:
# Convert the dictionary to a format shapely can work with
# The polygon coordinates are nested inside ['geometry']
df_green_spaces['geometry'] = df_green_spaces['geo_shape'].apply(
    lambda x: shape(x['geometry']))

In [ ]:
# Display geospatial columns for each record
df_green_spaces[['green_space_id', 'green_space_name', 'geo_shape', 'geo_point', 'lon', 'lat', 'geometry']].head()

,green_space_id,green_space_name,geo_shape,geo_point,lon,lat,geometry
0,13161,JARDINIERES EVQ DE LA RUE CRESPIN DU GAST,"{'type': 'Feature', 'geometry': {'coordinates'...","{'lon': 2.3818255187845714, 'lat': 48.86591139...",2.381826,48.865911,MULTIPOLYGON (((2.382027310280969 48.865601256...
1,13077,JARDINIERES DU SQUARE DE LA SALAMANDRE,"{'type': 'Feature', 'geometry': {'coordinates'...","{'lon': 2.4058613598651686, 'lat': 48.85790156...",2.405861,48.857902,MULTIPOLYGON (((2.4056850296912113 48.85823297...
2,11769,PROMENADE PC 12 - DAUMESNIL - SAHEL,"{'type': 'Feature', 'geometry': {'coordinates'...","{'lon': 2.406751045033287, 'lat': 48.838468901...",2.406751,48.838469,MULTIPOLYGON (((2.406449736575972 48.837624843...
3,237,SQUARE GUSTAVE MESUREUR,"{'type': 'Feature', 'geometry': {'coordinates'...","{'lon': 2.362371217923706, 'lat': 48.833740704...",2.362371,48.833741,POLYGON ((2.3628475557341475 48.83366797497178...
4,13147,JARDINS DES CHAMPS ELYSEES- SQUARE DE BERLIN- ...,"{'type': 'Feature', 'geometry': {'coordinates'...","{'lon': 2.31067213356462, 'lat': 48.8671491238...",2.310672,48.867149,POLYGON ((2.3104360759461775 48.86735334167561...


## 6. Export Dataset

In [ ]:
df_green_spaces.to_csv('../data/green_spaces.csv', index=False)
print("Dataset saved to '../data/green_spaces.csv'. You can now download it.")

Dataset saved to 'green_spaces.csv'. You can now download it.


In [ ]:
df_green_spaces.head()

,green_space_id,green_space_name,green_space_type,category,postal_code,polygon_area,opening_year,geo_shape,geo_point,lon,lat,geometry
0,13161,JARDINIERES EVQ DE LA RUE CRESPIN DU GAST,Street decorations,Planter box,75011,NaN,2026,"{'type': 'Feature', 'geometry': {'coordinates'...","{'lon': 2.3818255187845714, 'lat': 48.86591139...",2.381826,48.865911,MULTIPOLYGON (((2.382027310280969 48.865601256...
1,13077,JARDINIERES DU SQUARE DE LA SALAMANDRE,Street decorations,Planter box,75020,NaN,2021,"{'type': 'Feature', 'geometry': {'coordinates'...","{'lon': 2.4058613598651686, 'lat': 48.85790156...",2.405861,48.857902,MULTIPOLYGON (((2.4056850296912113 48.85823297...
2,11769,PROMENADE PC 12 - DAUMESNIL - SAHEL,Open promenades,Promenade,75012,11854.0,2019,"{'type': 'Feature', 'geometry': {'coordinates'...","{'lon': 2.406751045033287, 'lat': 48.838468901...",2.406751,48.838469,MULTIPOLYGON (((2.406449736575972 48.837624843...
3,237,SQUARE GUSTAVE MESUREUR,Open promenades,Public square,75013,3195.0,1982,"{'type': 'Feature', 'geometry': {'coordinates'...","{'lon': 2.362371217923706, 'lat': 48.833740704...",2.362371,48.833741,POLYGON ((2.3628475557341475 48.83366797497178...
4,13147,JARDINS DES CHAMPS ELYSEES- SQUARE DE BERLIN- ...,Open promenades,Garden,75008,NaN,1928,"{'type': 'Feature', 'geometry': {'coordinates'...","{'lon': 2.31067213356462, 'lat': 48.8671491238...",2.310672,48.867149,POLYGON ((2.3104360759461775 48.86735334167561...


In [ ]:
df_green_spaces.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2509 entries, 0 to 2527
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   green_space_id    2509 non-null   int64  
 1   green_space_name  2509 non-null   object 
 2   green_space_type  2509 non-null   object 
 3   category          2509 non-null   object 
 4   postal_code       2509 non-null   int64  
 5   polygon_area      1944 non-null   float64
 6   opening_year      1773 non-null   object 
 7   geo_shape         2509 non-null   object 
 8   geo_point         2509 non-null   object 
dtypes: float64(1), int64(2), object(6)
memory usage: 196.0+ KB
